# 05 — Cross-ethnic models

Does a model trained on one population work on another? Two directions are
trained and reported, and each is reported twice:

- **Within-cohort hold-out** — trained on 70% of a cohort, scored on the
  remaining 30%. This is the model's ceiling on its own population.
- **Cross-cohort** — the same model scored on the *entire* other cohort, which
  it has never seen. This is what transferability actually looks like.

The distinction matters because the legacy run folders `V31` and `V32` are
described as "train NHANES; test KNHANES" and vice versa, but the four lines that
would have swapped in the other cohort as the test set are commented out. Their
test sets are 3,498 and 4,542 rows — 30% of each cohort — so the numbers in
thesis Table `cross-ethnic1+2` are within-cohort hold-outs, not cross-ethnic
results. The genuinely cross-ethnic numbers are the ones in Table `cross-ethnic3`.
Both are produced here, each labelled for what it is.

Hyperparameters come from `configs/hyperparameters.yaml`. They were found once by
Bayesian optimisation and are not re-tuned on every run; the entry point for
re-running the search is at the bottom of this notebook.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import polars as pl

from src.data.io import load_hyperparameters, output_path, processed_path
from src.features.selection import drop_target_derived
from src.logging_utils import configure_logging
from src.models.evaluate import compute_metrics, score_external
from src.models.train import (
    build_classifier,
    build_voting_classifier,
    split_features_target,
    split_frames,
    stratified_split,
)

configure_logging(ROOT / "logs")

ALGORITHMS = ["CatBoost", "XGBoost", "lightGBM"]
HYPERPARAMETERS = load_hyperparameters()["cross_ethnic"]

cohorts = {
    "NHANES": drop_target_derived(
        pl.read_parquet(processed_path("NHANES_race1_features.parquet"))
    ),
    "KNHANES": drop_target_derived(
        pl.read_parquet(processed_path("KNHANES_race2_features.parquet"))
    ),
}
for name, frame in cohorts.items():
    print(f"{name}: {frame.shape}, IR+ {frame['IR'].mean():.4f}")

NHANES: (11660, 242), IR+ 0.4445
KNHANES: (15138, 242), IR+ 0.2792


## Within-cohort hold-out

Each cohort is split 70/30 with stratification, and three boosted-tree models
plus a soft-voting ensemble are trained on the training portion. The voting
ensemble refits its members, so it depends only on their hyperparameters and the
training data.

In [2]:
results = []
trained = {}

for cohort, frame in cohorts.items():
    params = HYPERPARAMETERS[cohort.lower()]
    X, y = split_features_target(frame)
    X_train, X_test, y_train, y_test = stratified_split(X, y)

    fitted = []
    for algorithm in ALGORITHMS:
        model = build_classifier(algorithm, y_train, params[algorithm])
        model.fit(X_train, y_train)
        fitted.append((algorithm, model))

    ensemble = build_voting_classifier(
        [(f"model_{index}", model) for index, (_, model) in enumerate(fitted)]
    )
    ensemble.fit(X_train, y_train)
    fitted.append(("Voting", ensemble))

    runs = {}
    for algorithm, model in fitted:
        preds = model.predict_proba(X_test)[:, 1]
        metrics = compute_metrics(y_test, preds)
        runs[algorithm] = {
            "model": model,
            "preds": preds,
            "metrics": metrics,
            "X_train": X_train,
            "X_test": X_test,
            "y_train": y_train,
            "y_test": y_test,
        }
        results.append(
            {
                "evaluation": "within-cohort hold-out",
                "trained_on": cohort,
                "evaluated_on": f"{cohort} (30% hold-out)",
                "model": algorithm,
                "n_samples": len(y_test),
                **{k: v for k, v in metrics.items() if k not in ("confusion_matrix", "optimal_threshold")},
                **metrics["confusion_matrix"],
            }
        )
    trained[cohort] = runs
    print(f"{cohort}: trained on {len(X_train):,} rows, scored on {len(X_test):,}")

NHANES: trained on 8,162 rows, scored on 3,498


KNHANES: trained on 10,596 rows, scored on 4,542


## Cross-cohort

The best model from each direction is scored on the **whole** of the other
cohort. The thesis reports CatBoost for NHANES and XGBoost for KNHANES, so those
are the two carried across.

In [3]:
CROSS_MODEL = {"NHANES": "CatBoost", "KNHANES": "XGBoost"}

for cohort, algorithm in CROSS_MODEL.items():
    other = "KNHANES" if cohort == "NHANES" else "NHANES"
    metrics = score_external(trained[cohort][algorithm]["model"], cohorts[other])
    results.append(
        {
            "evaluation": "cross-cohort",
            "trained_on": cohort,
            "evaluated_on": f"{other} (all)",
            "model": algorithm,
            "n_samples": metrics["n_samples"],
            **{
                k: v
                for k, v in metrics.items()
                if k not in ("confusion_matrix", "optimal_threshold", "preds", "n_samples")
            },
            **metrics["confusion_matrix"],
        }
    )

cross_ethnic = pl.DataFrame(results)
cross_ethnic.to_pandas().to_excel(output_path("cross_ethnic.xlsx"), index=False)
cross_ethnic

2026-09-14 20:54:34 [INFO] src.models.evaluate: External scoring on 15138 samples: AUC=0.862


2026-09-14 20:54:34 [INFO] src.models.evaluate: External scoring on 11660 samples: AUC=0.860


evaluation,trained_on,evaluated_on,model,n_samples,roc_auc,accuracy,sensitivity(recall),specificity,PPV(precision),NPV,f1_score,Youdens_Index,pr_auc,TP,TN,FP,FN
str,str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""CatBoost""",3498,0.868,0.788,0.76,0.81,0.762,0.808,0.761,0.57,0.85,1182,1573,370,373
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""XGBoost""",3498,0.867,0.787,0.758,0.81,0.761,0.807,0.76,0.568,0.848,1179,1573,370,376
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""lightGBM""",3498,0.867,0.79,0.759,0.814,0.766,0.808,0.762,0.573,0.851,1180,1582,361,375
"""within-cohort hold-out""","""NHANES""","""NHANES (30% hold-out)""","""Voting""",3498,0.868,0.79,0.758,0.815,0.766,0.808,0.762,0.573,0.851,1178,1584,359,377
"""within-cohort hold-out""","""KNHANES""","""KNHANES (30% hold-out)""","""CatBoost""",4542,0.878,0.798,0.781,0.805,0.608,0.905,0.683,0.586,0.767,990,2635,639,278
"""within-cohort hold-out""","""KNHANES""","""KNHANES (30% hold-out)""","""XGBoost""",4542,0.878,0.8,0.783,0.807,0.611,0.906,0.686,0.59,0.765,993,2642,632,275
"""within-cohort hold-out""","""KNHANES""","""KNHANES (30% hold-out)""","""lightGBM""",4542,0.877,0.797,0.78,0.803,0.606,0.904,0.682,0.583,0.766,989,2630,644,279
"""within-cohort hold-out""","""KNHANES""","""KNHANES (30% hold-out)""","""Voting""",4542,0.878,0.799,0.785,0.805,0.609,0.906,0.686,0.59,0.767,996,2635,639,272
"""cross-cohort""","""NHANES""","""KNHANES (all)""","""CatBoost""",15138,0.862,0.815,0.617,0.892,0.688,0.857,0.65,0.508,0.734,2607,9729,1183,1619


Transfer costs little in AUC — 0.868 → 0.862 one way, 0.878 → 0.860 the other —
but the operating point moves sharply. Scoring KNHANES with the NHANES model
trades sensitivity for specificity (0.760 → 0.617, 0.810 → 0.892); the reverse
direction does the opposite (0.783 → 0.890, 0.807 → 0.613). The threshold of 0.5
is calibrated to the training cohort's prevalence, which differs: 44% against
28%.

## Gate G5

Four families of check:

1. The hyperparameters in `configs/hyperparameters.yaml` against each run's JSON.
2. The training and test matrices, cell by cell. The `preds` column of the
   legacy `testing_data.csv` belongs to the **voting** model — `EnsembleModel`
   ran last in each legacy run and overwrote both CSVs.
3. All nine metrics plus the confusion matrix, for four models × two directions.
4. The cross-cohort metrics against thesis Table `cross-ethnic3`.

In [4]:
import json

from src.data.io import repo_path
from src.validate import compare_matrices, report

RUN_FOLDERS = {"NHANES": "20250221201739_V31", "KNHANES": "20250221205627_V32"}
METRIC_KEYS = [
    "roc_auc", "accuracy", "sensitivity(recall)", "specificity", "PPV(precision)",
    "NPV", "f1_score", "Youdens_Index", "pr_auc",
]
# Transcribed from tables/cross-ethnic3.tex, "Predicting" columns.
PUBLISHED_CROSS = {
    "NHANES": dict(zip(METRIC_KEYS, [0.862, 0.815, 0.617, 0.892, 0.688, 0.857, 0.650, 0.508, 0.734])),
    "KNHANES": dict(zip(METRIC_KEYS, [0.860, 0.736, 0.890, 0.613, 0.648, 0.875, 0.750, 0.503, 0.832])),
}

results_root = Path(repo_path("reference_results_root"))
passed = True

for cohort, folder in RUN_FOLDERS.items():
    run_dir = results_root / folder
    legacy = {
        algorithm: json.loads((run_dir / f"results_{algorithm}.json").read_text())
        for algorithm in ALGORITHMS + ["Voting"]
    }

    differences = [
        f"{algorithm}.{key}: {value!r} != published {legacy[algorithm]['best_params'][key]!r}"
        for algorithm in ALGORITHMS
        for key, value in HYPERPARAMETERS[cohort.lower()][algorithm].items()
        if value != legacy[algorithm]["best_params"][key]
    ]
    passed &= report(
        f"{cohort} [{folder}] configured hyperparameters",
        {"passed": not differences, "differences": differences},
    )

    # EnsembleModel ran last, so the exported CSVs carry the voting model's preds.
    training, testing = split_frames(trained[cohort]["Voting"])
    passed &= report(
        f"{cohort} [{folder}] training matrix",
        compare_matrices(training, pl.read_csv(run_dir / "training_data.csv")),
    )
    passed &= report(
        f"{cohort} [{folder}] test matrix and voting predictions",
        compare_matrices(testing, pl.read_csv(run_dir / "testing_data.csv")),
    )

    for algorithm in ALGORITHMS + ["Voting"]:
        metrics = trained[cohort][algorithm]["metrics"]
        differences = [
            f"{key}: {metrics[key]} != published {legacy[algorithm][key]}"
            for key in METRIC_KEYS
            if metrics[key] != legacy[algorithm][key]
        ] + [
            f"confusion_matrix.{key}: {metrics['confusion_matrix'][key]} != published {value}"
            for key, value in legacy[algorithm]["confusion_matrix"].items()
            if metrics["confusion_matrix"][key] != value
        ]
        passed &= report(
            f"{cohort} [{folder}] {algorithm} metrics",
            {"passed": not differences, "differences": differences},
        )

for cohort, algorithm in CROSS_MODEL.items():
    other = "KNHANES" if cohort == "NHANES" else "NHANES"
    row = cross_ethnic.filter(
        (pl.col("evaluation") == "cross-cohort") & (pl.col("trained_on") == cohort)
    ).to_dicts()[0]
    differences = [
        f"{key}: {row[key]} != published {value}"
        for key, value in PUBLISHED_CROSS[cohort].items()
        if row[key] != value
    ]
    passed &= report(
        f"cross-cohort {cohort} -> {other} ({algorithm}) vs thesis cross-ethnic3",
        {"passed": not differences, "differences": differences},
    )

print()
print("G5:", "PASS" if passed else "FAIL")

2026-09-14 20:54:35 [INFO] src.validate: PASS NHANES [20250221201739_V31] configured hyperparameters


2026-09-14 20:54:35 [INFO] src.validate: PASS NHANES [20250221201739_V31] training matrix


PASS  NHANES [20250221201739_V31] configured hyperparameters
PASS  NHANES [20250221201739_V31] training matrix


2026-09-14 20:54:35 [INFO] src.validate: PASS NHANES [20250221201739_V31] test matrix and voting predictions


2026-09-14 20:54:35 [INFO] src.validate: PASS NHANES [20250221201739_V31] CatBoost metrics


...

2026-09-14 20:54:36 [INFO] src.validate: PASS KNHANES [20250221205627_V32] lightGBM metrics


2026-09-14 20:54:36 [INFO] src.validate: PASS KNHANES [20250221205627_V32] Voting metrics


2026-09-14 20:54:37 [INFO] src.validate: PASS cross-cohort NHANES -> KNHANES (CatBoost) vs thesis cross-ethnic3


2026-09-14 20:54:37 [INFO] src.validate: PASS cross-cohort KNHANES -> NHANES (XGBoost) vs thesis cross-ethnic3


PASS  KNHANES [20250221205627_V32] configured hyperparameters
PASS  KNHANES [20250221205627_V32] training matrix
PASS  KNHANES [20250221205627_V32] test matrix and voting predictions
PASS  KNHANES [20250221205627_V32] CatBoost metrics
PASS  KNHANES [20250221205627_V32] XGBoost metrics
PASS  KNHANES [20250221205627_V32] lightGBM metrics
PASS  KNHANES [20250221205627_V32] Voting metrics
PASS  cross-cohort NHANES -> KNHANES (CatBoost) vs thesis cross-ethnic3
PASS  cross-cohort KNHANES -> NHANES (XGBoost) vs thesis cross-ethnic3

G5: PASS


## Re-running the hyperparameter search

Off by default. The published models were tuned once and their parameters
committed; this exists so the search can be repeated, not so it runs every time.
Set `RETUNE = True` to tune one combination and compare the result against the
committed values.

In [5]:
RETUNE = False

if RETUNE:
    from src.models.train import tune_hyperparameters

    cohort, algorithm = "NHANES", "XGBoost"
    X, y = split_features_target(cohorts[cohort])
    X_train, _, y_train, _ = stratified_split(X, y)

    recovered = tune_hyperparameters(X_train, y_train, algorithm)
    committed = HYPERPARAMETERS[cohort.lower()][algorithm]
    for key in sorted(set(recovered) | set(committed)):
        mark = "==" if recovered.get(key) == committed.get(key) else "!="
        print(f"{key:20s} {recovered.get(key)!r} {mark} {committed.get(key)!r}")